**Objective**: This notebook introduces the initial Exploratory Data Analysis (EDA) process that was conducted, as well as the development of a baseline text classification model.

# 1. Exploratory Data Analysis

This first section will focus on the understanding of the data provided in order to get a global overview of it while also identifying potential patterns, imbalances or other factos that may influence the classification task at hand.

**Step 1:** Importing the necessary libraries and the dataset. It is important to note that only the training data will be imported since the test set will only be used for the classification of the final model and, therefore, it is not needed in this notebook.

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [37]:
train_data = pd.read_csv("data/train.csv")

In [38]:
train_data.head()

,title,tag,artist,year,views,features,lyrics,id
0,Walk Away,rock,Tony Molina,2013,699,{},When you said you loved me\nDid you mean it th...,535805
1,Gotta Make It Kid Naruto Rap,rap,Reece Lett,2021,4,{Sl!ck},Kid Naruto Rap\n[Hook]\nEverybody wants you to...,7519483
2,​this is what i asked for,pop,Elliot (DNK),2019,389,{},[Verse 1]\nPeople tell me I've changed\nI find...,4892808
3,Stealing Hearts,pop,Katie Armiger,2013,126,{},You've been warned about me\nDon't try to get ...,1584150
4,Get Ready,country,John Campbell Munro,1,2,{},[Verse 1]\nI can see the end is coming but I’v...,7639050


**NOTE**: For the meaning of each feature, please refer to the *README.md* file.

**Step 2**: Understanding the structure of the dataset.

In [39]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 134967 entries, 0 to 134966
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   title     134965 non-null  str  
 1   tag       134967 non-null  str  
 2   artist    134967 non-null  str  
 3   year      134967 non-null  int64
 4   views     134967 non-null  int64
 5   features  134967 non-null  str  
 6   lyrics    134967 non-null  str  
 7   id        134967 non-null  int64
dtypes: int64(3), str(5)
memory usage: 8.2 MB


Based on the overview above, we can already identify some missing values (2) on the `title` column, with the remaining features complete (note that {} is a placeholder for missing features). This dataset has 134967 obvervations and 8 attributes, with 3 of them being numerical and the remaining 5 being string columns.

**Step 3**: Checking for duplicate `id`s.

In [40]:
train_data['id'].is_unique

True

There are no duplicates on the `id` column. Therefore, it can be set as the index of our dataframe.

In [41]:
train_data.set_index('id', inplace = True)

In [42]:
train_data

,title,tag,artist,year,views,features,lyrics
id,,,,,,,
535805,Walk Away,rock,Tony Molina,2013,699,{},When you said you loved me\nDid you mean it th...
7519483,Gotta Make It Kid Naruto Rap,rap,Reece Lett,2021,4,{Sl!ck},Kid Naruto Rap\n[Hook]\nEverybody wants you to...
4892808,​this is what i asked for,pop,Elliot (DNK),2019,389,{},[Verse 1]\nPeople tell me I've changed\nI find...
1584150,Stealing Hearts,pop,Katie Armiger,2013,126,{},You've been warned about me\nDon't try to get ...
7639050,Get Ready,country,John Campbell Munro,1,2,{},[Verse 1]\nI can see the end is coming but I’v...
...,...,...,...,...,...,...,...
1702980,Manhattan,pop,Tijuana Sweetheart,2007,48,{},If I knew when I was young that I'd be older\n...
4802970,Belly Shit,rap,Lil Gotit,2019,3593,"{""Lil Troup""}",[Intro: Lil Gotit]\nCash\nWah-wah-wah\nWah-wah...
403929,The Four Loves,rock,Heath McNease,2014,301,{},[Verse 1]\nI was born inside a home\nThe young...


**Step 4**: Checking for duplicate songs on the dataset (same title, artist and lyrics).

In [43]:
bool(train_data.duplicated(subset=['title', 'artist', 'lyrics']).any())

False

**Step 5**: Analysing the target variable (`tag`).

In [44]:
display(train_data['tag'].value_counts())
display(train_data['tag'].value_counts(normalize = True) * 100)

tag
pop        55742
rap        38584
rock       25332
rb          6203
misc        5640
country     3466
Name: count, dtype: int64

tag
pop        41.300466
rap        28.587729
rock       18.769032
rb          4.595938
misc        4.178799
country     2.568035
Name: proportion, dtype: float64

By analysing the ouput, we can see that the songs can be classified into **6** genres: **pop**, **rap**, **rock**, **R&B (rb)**, **miscellaneous (misc)** and **country**. The distribution of this genres across the dataset is imbalanced, with **pop** being the **most** frequent genre (41.3 %) and **country** the **least** frequent (only 2.57%). It is important to acknowledge this imbalance when dealing with a text classification task since the model might become biased towards the more representative class and fail to classify correctly the minority class. Therefore, the model evaluation process should consider metrics like the macro F1-score, instead of simply using the accuracy, to correctly assess performance across all genres.

**Step 6**: Analyse the numerical features.

In [45]:
train_data.describe()

,year,views
count,134967.000000,1.349670e+05
mean,2009.467537,3.413344e+03
std,46.287743,4.192449e+04
min,1.000000,0.000000e+00
25%,2008.000000,2.200000e+01
50%,2015.000000,8.900000e+01
75%,2019.000000,4.730000e+02
max,2024.000000,3.604497e+06


As we can see, there is an issue with the year column: the minimum value for the year ("the oldest year") is 1. This value will be treated as an error on the dataset.

In [47]:
train_data[train_data['year'] == 1]

,title,tag,artist,year,views,features,lyrics
id,,,,,,,
7639050,Get Ready,country,John Campbell Munro,1,2,{},[Verse 1]\nI can see the end is coming but I’v...
113408,Acts 3,misc,Holy Bible (KJV),1,417,{},Acts 3:1: Now Peter and John went up together ...
7361529,Lake Michigan,rap,DON KEL,1,6,{},I don't fuck with many n**** (Woah)\nI be on m...
6905467,Secondhand Clothes,rock,Maker of the Bear,1,17,{},Verse 1\nI was talking to myself\nFull of fear...
6125089,ADW,rap,Lik2ss,1,624,{},"Intro : 2ss! 2ss\n42uce DreamTeam Nigga, Free ..."
3836184,Woop,rap,Lxrry,1,15,{},[Intro]: Lxrry\nWe taking over like woop\n\nWe...
7214560,Sick Psycho 6ircus,rock,Inkwhore333,1,15,"{""Keyawna Nikole""}",Come here lil bitch think im losing my patienc...
108868,Genesis 28,misc,Holy Bible (KJV),1,2045,{},"Genesis 28:1: And Isaac called Jacob, and bles..."
5600941,Most Days,rb,Peter Fenn,1,596,{},I don't want nothing to do with love\nOn most ...
